# RSNA Knee — ensemble inference (internet OFF)

| Experiment | Notebook | What this version does |
|---|---|---|
| **EXP-104** | `hlly34/rsna-knee-inference` | Rule-safe (competition-data models only): 96-slice seeds 0.35+0.35 + reader 5f 0.15 (mirror) + goodpjw public reader 0.15; no nartaa / heliosli |

Scores the hidden test set with every fold model attached from the weights dataset `Kaggle Models: rsna-knee-raptor-v2-d96, rsna-knee-reader-repro` (uploaded by `scripts/vast/submit.sh`). Test DICOMs go through exactly the same preprocessing as training (`src/dataset/dicom.py`); each model's predictions are converted to per-finding ranks (AUC only depends on order) and averaged.

Pipeline code lives in the attached dataset `rsna-knee-code` (repo `src/`, `configs/`, `scripts/`); this notebook only orchestrates it. Results are registered with `scripts/track.py` into `experiments/registry.csv` using the id above.

In [ ]:
EXP = {"id": "EXP-104", "notebook": "hlly34/rsna-knee-inference", "what": "Rule-safe (competition-data models only): 96-slice seeds 0.35+0.35 + reader 5f 0.15 (mirror) + goodpjw public reader 0.15; no nartaa / heliosli"}
print("EXPERIMENT", EXP["id"], "-", EXP["what"])
FAMILY_WEIGHTS = 'raptor_v2_d96_alldata=0.35,raptor_v2_d96_alldata_s2=0.35,reader_repro=0.15'
INFER_ARGS = ['--mirror_tta']
EXT_WEIGHT = 0.0   # rank weight of the external nartaa 0.949 stage (None = off)
EXT_MAP = {}   # per-finding override of EXT_WEIGHT (finding-group routing)
EXT224 = None   # share of nartaa's 0.945 224-crop model inside the external anchor (None = off)
EXT_HEL = None   # rank weight of heliosli's public compact ConvNeXt-S stage (None = off)
FUSION = 'rank'   # rank: average percentile ranks | probit: average their normal quantiles
EXT_GPJ = 0.15   # rank weight of goodpjw's public 2.5D ConvNeXt reader stage (None = off)
EXT_GPJ_MAP = {}   # per-finding goodpjw weight (overrides EXT_GPJ for those findings)
EXT_HEL_SCRIPT = 'heliosli_compact_stage.py'   # heliosli stage: compact v1 (S6) or R7
import glob, json, os, subprocess, sys, time

def walk_find(pred, root="/kaggle/input", max_depth=7):
    """Depth-limited search of the attached inputs that never descends into the DICOM trees."""
    root = root.rstrip("/")
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath[len(root):].count(os.sep)
        dirnames[:] = [] if depth >= max_depth else sorted(d for d in dirnames if not d.endswith("_series"))
        hit = pred(dirpath, filenames)
        if hit:
            return hit
    return None

# Kaggle mounts inputs at /kaggle/input/{competitions,datasets,notebooks}/... - locate them instead of hardcoding
CODE = walk_find(lambda d, f: d if os.path.isfile(os.path.join(d, "src", "models", "inference.py")) else None)
COMP = walk_find(lambda d, f: d if "test.csv" in f and os.path.isdir(os.path.join(d, "test_series")) else None)
assert CODE and COMP, (CODE, COMP)
sys.path.insert(0, CODE)
print("code:", CODE, "| competition:", COMP)


## 1. Offline dependencies and checkpoints
The pylibjpeg wheels come from a training notebook's output (needed for JPEG-compressed DICOMs).

In [ ]:
WHEELS = walk_find(lambda d, f: d if any(x.startswith("pylibjpeg") and x.endswith(".whl") for x in f) else None)
if WHEELS:
    # --no-deps: never let an offline wheel (e.g. numpy) replace the image's own packages
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps", "--find-links", WHEELS,
                    "pylibjpeg", "pylibjpeg-libjpeg", "pylibjpeg-openjpeg"], check=False)
CKPTS = []
# our checkpoints only: the external stage's weights (no cfg inside) are run by its own script
EXTERNAL_DIRS = ("rsna-knee-publication-swa-weights", "rsna-knee-compact-v1", "rsna-knee-compact-r7", "rsna-orthofoundation",
                 "rsna-knee-2-5d-convnext-reader")  # run by own scripts
walk_find(lambda d, f: CKPTS.extend(os.path.join(d, x) for x in sorted(f) if x.endswith(".pt")
                                    and not any(e in d for e in EXTERNAL_DIRS)) and None)
print("wheels:", WHEELS)
try:
    import pylibjpeg, libjpeg, openjpeg  # noqa: F401  JPEG / JPEG-2000 DICOM decoders
    print("JPEG decoders: OK")
except ImportError as e:
    print("JPEG decoders MISSING:", e)
for c in CKPTS: print(" ", c)
assert CKPTS, "attach the training notebooks' outputs"

## 1b. External stage — nartaa's 0.949 CoAtNet (verbatim scored source, OAI-trained; fallback pick)

In [ ]:
import shutil
t_ext = time.time()
NARTAA_ON = bool(EXT_WEIGHT) or any(float(v) for v in EXT_MAP.values())
if NARTAA_ON:
    subprocess.run([sys.executable, f"{CODE}/submission/external/nartaa_0949_check.py"], check=True)
    subprocess.run([sys.executable, f"{CODE}/submission/external/nartaa_0949_inference.py"], check=True, cwd="/kaggle/working")
    shutil.move("/kaggle/working/submission.csv", "/kaggle/working/ext_nartaa.csv")
if NARTAA_ON and EXT224:  # second view of the anchor: nartaa's 224-crop efficiency checkpoint (verbatim scored source)
    subprocess.run([sys.executable, f"{CODE}/submission/external/nartaa_0945_check.py"], check=True)
    subprocess.run([sys.executable, f"{CODE}/submission/external/nartaa_0945_inference.py"], check=True, cwd="/kaggle/working")
    shutil.move("/kaggle/working/submission.csv", "/kaggle/working/ext_nartaa224.csv")
if EXT_HEL:  # heliosli's public compact ConvNeXt-S model (verbatim stage, writes ours_sub.csv)
    subprocess.run([sys.executable, f"{CODE}/submission/external/{EXT_HEL_SCRIPT}"], check=True, cwd="/kaggle/working")
    shutil.move("/kaggle/working/ours_sub.csv", "/kaggle/working/ext_heliosli.csv")
if EXT_GPJ or EXT_GPJ_MAP:  # goodpjw's public 2.5D ConvNeXt reader (verbatim source), averaged over its 3 public folds
    GPJ_CK = []
    walk_find(lambda d, f: GPJ_CK.extend(os.path.join(d, x) for x in sorted(f)
                                         if x.startswith("cnxt_") and x.endswith(".pt")) and None)
    assert GPJ_CK, "goodpjw reader weights not attached"
    print("goodpjw reader ckpts:", GPJ_CK)
    subprocess.run([sys.executable, f"{CODE}/submission/external/goodpjw_reader/infer.py", COMP, "test",
                    "/kaggle/working/ext_goodpjw.csv", *GPJ_CK], check=True, cwd=f"{CODE}/submission/external/goodpjw_reader")
EXT_SECONDS = round(time.time() - t_ext)
print(f"external stage {EXT_SECONDS}s")

## 2. Predict and rank-average

In [ ]:
t0 = time.time()
from src.models.inference import main
sys.argv = ["inference", "--data_dir", COMP, "--ckpts", *CKPTS, "--rank_average", "--cache_dir", "/tmp/knee_cache",
            *(["--family_weights", FAMILY_WEIGHTS] if FAMILY_WEIGHTS else []), *INFER_ARGS,
            "--workers", str(os.cpu_count()), "--out", "/kaggle/working/submission.csv"]
main()
print(f"inference {time.time() - t0:.0f}s")

In [ ]:
import pandas as pd
# rank fusion: EXT_WEIGHT x nartaa (external) + (1 - EXT_WEIGHT) x our blend, per finding, then re-rank
ours = pd.read_csv("/kaggle/working/submission.csv", dtype={"StudyInstanceUID": str}).set_index("StudyInstanceUID")
from scipy.stats import norm


def tf(r):  # rank -> fusion space (probit keeps the extremes apart instead of compressing them toward the middle)
    return pd.DataFrame(norm.ppf(r.clip(1e-4, 1 - 1e-4)), index=r.index, columns=r.columns) if FUSION == "probit" else r


if NARTAA_ON:
    ext = pd.read_csv("/kaggle/working/ext_nartaa.csv", dtype={"StudyInstanceUID": str}).set_index("StudyInstanceUID")
    assert set(ours.index) == set(ext.index), "external and our predictions cover different studies"
    anchor = tf(ext.loc[ours.index, ours.columns].rank(pct=True))
else:  # rule-safe blend: no nartaa anchor
    anchor = tf(ours.rank(pct=True)) * 0.0
if NARTAA_ON and EXT224:
    e224 = pd.read_csv("/kaggle/working/ext_nartaa224.csv", dtype={"StudyInstanceUID": str}).set_index("StudyInstanceUID")
    assert set(e224.index) == set(ours.index), "224-crop predictions cover different studies"
    anchor = (1 - EXT224) * anchor + EXT224 * tf(e224.loc[ours.index, ours.columns].rank(pct=True))
w = pd.Series({c: float(EXT_MAP.get(c, EXT_WEIGHT or 0.0)) for c in ours.columns})   # nartaa share per finding
wh = float(EXT_HEL or 0.0)
wg = pd.Series({c: float(EXT_GPJ_MAP.get(c, EXT_GPJ or 0.0)) for c in ours.columns})   # goodpjw share per finding
assert (w + wh + wg <= 1 + 1e-9).all(), "external weights exceed 1"
fused = anchor.mul(w, axis=1) + tf(ours.rank(pct=True)).mul(1 - w - wh - wg, axis=1)
if wg.any():
    gpj = pd.read_csv("/kaggle/working/ext_goodpjw.csv", dtype={"StudyInstanceUID": str}).set_index("StudyInstanceUID")
    assert set(gpj.index) == set(ours.index), "goodpjw predictions cover different studies"
    fused = fused + tf(gpj.loc[ours.index, ours.columns].rank(pct=True)).mul(wg, axis=1)
if wh:
    hel = pd.read_csv("/kaggle/working/ext_heliosli.csv", dtype={"StudyInstanceUID": str}).set_index("StudyInstanceUID")
    assert set(hel.index) == set(ours.index), "heliosli predictions cover different studies"
    fused = fused + wh * tf(hel.loc[ours.index, ours.columns].rank(pct=True))
fused = fused.rank(pct=True)
assert fused.notna().all().all()
fused.reset_index().to_csv("/kaggle/working/submission.csv", index=False)
print("fused", fused.shape, "| ext weight per finding", w.round(3).to_dict(), "| ext224", EXT224, "| heliosli", EXT_HEL, "| goodpjw", wg.round(3).to_dict(), "| fusion", FUSION)

In [ ]:
import numpy as np, pandas as pd
sub = pd.read_csv("/kaggle/working/submission.csv")
files = glob.glob("/tmp/knee_cache/*/*.npz")   # only written in --mode cache (stream keeps stacks in memory)
EXP.update(models=len(CKPTS), test_studies=len(sub), seconds=round(time.time() - t0),
           ckpts=[os.path.join(os.path.basename(os.path.dirname(c)), os.path.basename(c)) for c in CKPTS],
           ckpt_paths=CKPTS, family_weights=FAMILY_WEIGHTS, ext_weight=EXT_WEIGHT,
           ext_seconds=globals().get("EXT_SECONDS"),
           upload_ids=[])
walk_find(lambda d, f: EXP["upload_ids"].extend(sorted(x for x in f if x.startswith("UPLOAD_"))) and None)
if files:
    EXP["test_slot_coverage"] = round(float(np.mean([np.load(f)["valid"].mean() for f in files])), 4)
json.dump(EXP, open("/kaggle/working/experiment.json", "w"), indent=1)
print(EXP); sub.head()